# Исследование территории Светлогорска с помощью BlocksNet

Учебный ноутбук для анализа территории **Светлогорск, Калининградская область, Россия**.
Все расчёты выполняются с помощью библиотеки **BlocksNet 1.0.0.a9**.

## Что делает этот ноутбук

1. Загружает границу территории из OpenStreetMap (автоматический поиск).
2. Готовит городские объекты (дороги, железные дороги, водные объекты).
3. Режет территорию на городские кварталы.
4. ⏸ **ОСТАНОВКА** — перед расчётами уточняются доступные метрики.
5. Назначает земельные 이용ы кварталам.
6. Строит пешеходный граф и считает матрицу времени в пути между кварталами (в минутах).
7. Агрегирует здания и сервисы (например, школы) в кварталы.
8. Считает все показатели из `pipeline.ipynb`: обеспечение, плотность, морфотипы, доступность территории.
9. Сохраняет промежуточные и итоговые данные в GeoJSON и CSV.

## Важное предупреждение

Ноутбук **не скачивает данные и не запускает расчёты автоматически**. Вы запускаете ячейки вручную и просматриваете результаты.
Перед запуском расчётных ячеек (после подготовки кварталов) уточните у автора, какие метрики необходимы.

In [1]:
# =============================================================================
# 0. Настройка и импорты
# =============================================================================

import os
from pathlib import Path

import geopandas as gpd
import osmnx as ox
import pandas as pd

from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.enums import LandUse
from blocksnet.relations import get_accessibility_graph, calculate_accessibility_matrix
from blocksnet.preprocessing.imputing import impute_buildings, impute_services
from blocksnet.config import service_types_config
from blocksnet.analysis.provision import competitive_provision
from blocksnet.analysis.indicators import calculate_density_indicators
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes
from blocksnet.analysis.network.accessibility import area_accessibility

# --- Конфигурация территории ---
TERRITORY_NAME = "Светлогорск, Калининградская область, Россия"
GRAPH_TYPE = "walk"  # пешеходный граф (не интермодальный)
TIME_UNIT = "минуты"

# --- Папки ---
BASE_DIR = Path.cwd()  # Папка, из которой запущен ноутбук (должна быть examples/)
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# --- Пути к файлам (GeoJSON / CSV) ---
BOUNDARY_PATH = RESULTS_DIR / "boundary.geojson"
BLOCKS_PATH = RESULTS_DIR / "blocks.geojson"
BLOCKS_LANDUSE_PATH = RESULTS_DIR / "blocks_landuse.geojson"
ACCESSIBILITY_MX_PATH = RESULTS_DIR / "accessibility_matrix.csv"
BUILDINGS_PATH = RESULTS_DIR / "buildings.geojson"
BUILDINGS_BLOCKS_PATH = RESULTS_DIR / "buildings_blocks.geojson"
SERVICES_PATH = RESULTS_DIR / "services.geojson"
SERVICES_BLOCKS_PATH = RESULTS_DIR / "services_blocks.geojson"
PROVISION_PATH = RESULTS_DIR / "provision.csv"
DENSITY_PATH = RESULTS_DIR / "density_indicators.csv"
MORPHOTYPES_PATH = RESULTS_DIR / "morphotypes.csv"
AREA_ACCESSIBILITY_PATH = RESULTS_DIR / "area_accessibility.csv"
LINKS_PATH = RESULTS_DIR / "links.csv"

print(f"Территория: {TERRITORY_NAME}")
print(f"Тип графа: {GRAPH_TYPE} (пешеходный)")
print(f"Единица времени: {TIME_UNIT}")
print(f"Папка данных: {DATA_DIR}")
print(f"Папка результатов: {RESULTS_DIR}")


d:\ИТМО\AIcourseNAT\python-package-boilerplate\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


ModuleNotFoundError: No module named 'blocksnet.analysis.accessibility'

## 1. Резка городских кварталов (Blocks cutting)

### 1.1. Загрузка границы территории из OpenStreetMap

Шаг можно пропустить, если файл с границей уже существует в `results/boundary.geojson`.

In [ ]:
# =============================================================================
# 1.1. Загрузка границы территории
# =============================================================================

if BOUNDARY_PATH.exists():
    print(f"Загрузка существующей границы из {BOUNDARY_PATH}")
    boundaries = gpd.read_file(BOUNDARY_PATH)
else:
    print(f"Поиск границы через OSM: {TERRITORY_NAME}")
    boundaries = ox.geocode_to_gdf(TERRITORY_NAME)
    boundaries.to_file(BOUNDARY_PATH, driver="GeoJSON")
    print(f"Граница сохранена в {BOUNDARY_PATH}")

print(f"Количество объектов границы: {len(boundaries)}")
print(f"CRS: {boundaries.crs}")
boundaries.head()

In [ ]:
# Визуализация границы
boundaries.plot().set_axis_off()

### 1.2. Предобработка городской геометрии

Загружаем дороги, железные дороги и водные объекты из OpenStreetMap,
фильтруем геометрии и приводим все слои к единой системе координат (СК).
Промежуточные файлы сохраняются в `data/` и при повторном запуске загружаются из них.

In [ ]:
# =============================================================================
# 1.2. Предобработка городских объектов
# =============================================================================

bc_tags = {
    "roads": {
        "highway": [
            "construction", "crossing", "living_street", "motorway",
            "motorway_link", "motorway_junction", "pedestrian", "primary",
            "primary_link", "raceway", "residential", "road", "secondary",
            "secondary_link", "services", "tertiary", "tertiary_link",
            "track", "trunk", "trunk_link", "turning_circle", "turning_loop",
            "unclassified",
        ],
        "service": ["living_street", "emergency_access"],
    },
    "railways": {
        "railway": "rail",
    },
    "water": {
        "riverbank": True,
        "reservoir": True,
        "basin": True,
        "dock": True,
        "canal": True,
        "pond": True,
        "natural": ["water", "bay"],
        "waterway": ["river", "canal", "ditch"],
        "landuse": "basin",
        "water": "lake",
    },
}

crs = boundaries.estimate_utm_crs()
print(f"Используемая СК: {crs}")

water_path = DATA_DIR / "water.geojson"
roads_path = DATA_DIR / "roads.geojson"
railways_path = DATA_DIR / "railways.geojson"

# Водные объекты
if water_path.exists():
    water = gpd.read_file(water_path)
    print(f"Водные объекты загружены из {water_path}")
else:
    water = ox.features_from_polygon(boundaries.union_all(), bc_tags["water"])
    water = water[water.geom_type.isin(["Polygon", "MultiPolygon", "LineString", "MultiLineString"])].copy()
    water.to_file(water_path, driver="GeoJSON")
    print(f"Водные объекты сохранены в {water_path}")

# Дороги
if roads_path.exists():
    roads = gpd.read_file(roads_path)
    print(f"Дороги загружены из {roads_path}")
else:
    roads = ox.features_from_polygon(boundaries.union_all(), bc_tags["roads"])
    roads = roads[roads.geom_type.isin(["LineString", "MultiLineString"])].copy()
    roads.to_file(roads_path, driver="GeoJSON")
    print(f"Дороги сохранены в {roads_path}")

# Железные дороги
if railways_path.exists():
    railways = gpd.read_file(railways_path)
    print(f"Железные дороги загружены из {railways_path}")
else:
    railways = ox.features_from_polygon(boundaries.union_all(), bc_tags["railways"])
    railways = railways[railways.geom_type.isin(["LineString", "MultiLineString"])].copy()
    railways.to_file(railways_path, driver="GeoJSON")
    print(f"Железные дороги сохранены в {railways_path}")

# Приведение к общей СК
for gdf in [water, roads, railways, boundaries]:
    gdf.to_crs(crs, inplace=True)

print(f"Водных объектов: {len(water)}")
print(f"Дорог: {len(roads)}")
print(f"Железных дорог: {len(railways)}")

### 1.3. Резка городских кварталов

Разделение территории на кварталы с использованием дорог, железных дорог и водных объектов как границ.

⏸ **ОСТАНОВКА**: После этого шага перед запуском расчётов уточните, какие метрики вам необходимы.

In [ ]:
# =============================================================================
# 1.3. Резка городских кварталов
# =============================================================================

if BLOCKS_PATH.exists():
    print(f"Загрузка существующих кварталов из {BLOCKS_PATH}")
    blocks = gpd.read_file(BLOCKS_PATH)
else:
    lines, polygons = preprocess_urban_objects(roads, railways, water)
    blocks = cut_urban_blocks(boundaries, lines, polygons)
    blocks.to_file(BLOCKS_PATH, driver="GeoJSON")
    print(f"Кварталы сохранены в {BLOCKS_PATH}")

print(f"Количество кварталов: {len(blocks)}")
blocks.head()

In [ ]:
# Визуализация кварталов
blocks.plot().set_axis_off()

## ⏸ ОСТАНОВКА ПОДГОТОВКИ КВАРТАЛОВ

Кварталы территории Светлогорска подготовлены и сохранены в `results/blocks.geojson`.

Перед запуском следующих шагов (назначение земельных 이용ов, расчёт матрицы доступности,
агрегация зданий и сервисов, расчёт показателей) **уточните, какие метрики вам необходимы**.

Доступные группы метрик (обсуждаются отдельно):
- конкурентное обеспечение сервисами (`competitive_provision`)
- плотностные индикаторы (`calculate_density_indicators`)
- морфотипы пространственной матрицы (`get_spacematrix_morphotypes`, `get_strelka_morphotypes`)
- доступность территории (`area_accessibility`)

## 📎 Источники данных, единицы измерения и ограничения

### Источники данных

- **Граница территории**: OpenStreetMap через `ox.geocode_to_gdf()` — поиск по названию "Светлогорск, Калининградская область, Россия"
- **Городские объекты (дороги, железные дороги, водные объекты)**: OpenStreetMap через `ox.features_from_polygon()`
- **Пешеходный граф**: OpenStreetMap через библиотеку `iduedu` (входит в состав BlocksNet), тип графа `walk`
- **Здания**: OpenStreetMap, тег `building`
- **Сервисы (школы)**: OpenStreetMap, тег `amenity=school`
- **Библиотека**: BlocksNet 1.0.0.a9

### Единицы измерения

- Время в пути: **минуты**
- Площадь: **кв.м**
- Население: **человек**
- Вместимость школ: **человек**
- Коэффициенты (FSI, GSI, MXI, L, OSR, provision, доступность): **без размерности (коэф. 0–1 или более)**

### Ограничения

- Данные OpenStreetMap могут быть неполными для территорий с низкой активностью маппинга
- Восстановленные значения (через `impute_buildings`, `impute_services`) — это **оценки**, а не исходные измерения
- Пешеходный граф не учитывает рельеф, перекрытия, закрытые территории
- Матрица доступности считается по центроидам кварталов

### Отметка об оценках

Следующие колонки содержат восстановленные (оценочные) значения, а не исходные данные OSM:
- `is_living`, `number_of_floors`, `footprint_area`, `build_floor_area`, `living_area`, `non_living_area`, `population` — после `impute_buildings()`
- `capacity` — после `impute_services()`


## 📊 Все показатели из pipeline.ipynb — простыми словами

Ниже перечислены все метрики, которые считаются в этом ноутбуке. Они помогут понять структуру территории Светлогорска и качество городской среды.

### 1. Матрица времени в пути (Accessibility matrix)

Таблица размером n×n (где n — количество кварталов). В каждой ячейке указано, сколько минут пешком идти из одного квартала в другой.

- **Единица:** минуты
- **Что покажет:** какие кварталы находятся близко, а которые далеко; где возможна транспортная изоляция или, наоборот, хорошая связность.

### 2. Параметры зданий (Buildings aggregation)

Агрегированные по кварталам характеристики зданий:

- **population** — количество жителей в квартале (человек)
- **footprint_area** — площадь застройки зданий (кв.м)
- **build_floor_area** — суммарная площадь всех этажей всех зданий (кв.м)
- **living_area** — жилая площадь (кв.м)
- **non_living_area** — нежилая площадь (кв.м)
- **objects_count** — количество зданий в квартале (шт.)

- **Что покажет:** где больше людей, как застроена территория, где преобладает жильё, а где нежилые здания.

### 3. Параметры сервисов (Services aggregation)

Агрегированные по кварталам характеристики сервисных объектов (например, школ):

- **capacity** — вместимость сервиса (например, сколько мест в школе — человек)
- **objects_count** — количество объектов сервиса в квартале (шт.)

- **Что покажет:** где расположены школы и другие объекты, какая у них вместимость.

### 4. Конкурентное обеспечение (Competitive provision)

Показывает, насколько хорошо жители квартала обеспечены сервисами (например, школами) в шаговой доступности, учитывая время в пути и конкуренцию между жителями разных кварталов.

- **demand** — спрос: сколько жителей нуждаются в сервисе (человек)
- **capacity** — oferta: сколько мест предоставляют сервисы в квартале (человек)
- **demand_left** — неудовлетворённый спрос после обеспечения внутри квартала (человек)
- **demand_within** — спрос, закрытый сервисами внутри этого же квартала (человек)
- **demand_without** — спрос, закрытый сервисами в других кварталах (человек)
- **capacity_left** — неиспользованная вместимость сервисов (человек)
- **capacity_within** — вместимость, использована жителями этого квартала (человек)
- **capacity_without** — вместимость, использована жителями других кварталов (человек)
- **provision_strong** — сильное обеспечение (от 0 до 1): доля спроса, которую можно закрыть сервисами в доступной зоне
- **provision_weak** — слабое обеспечение (от 0 до 1): более консервативная оценка обеспечения

- **Единица:** человек для абсолютных величин; коэффициент от 0 до 1 для provision_strong и provision_weak
- **Что покажет:** обеспечены ли жители квартала школами в шаговой доступности. 0 = совсем нет обеспечения, 1 = полное обеспечение.

### 5. Плотностные индикаторы (Density indicators)

Показатели плотности и этажности застройки кварталов:

- **site_area** — площадь квартала (кв.м)
- **fsi** — коэффициент плотности застройки (FSI): отношение суммарной площади всех этажей к площади участка (без размерности)
- **gsi** — коэффициент застройки (GSI): отношение площади застройки к площади участка (без размерности)
- **mxi** — коэффициент этажности (MXI): среднее количество этажей (без размерности)
- **l** — коэффициент этажности (L): отношение жилой площади к площади участка (без размерности)
- **osr** — коэффициент открытых пространств (OSR): доля незастроенной территории (без размерности)
- **share_living** — доля жилой площади (без размерности)
- **share_non_living** — доля нежилой площади (без размерности)

- **Что покажет:** как плотно застроены кварталы, какой этажности дома, сколько места остаётся под дворы, улицы и зелёные зоны.

### 6. Морфотипы застройки (Morphotypes)

Классификация кварталов по типам городской застройки:

- **Пространственная матрица (spacematrix):** кварталы делятся на типы, например: низкоэтажная нежилая, среднеэтажная жилая, высокоэтажная жилая и др.
- **Стрелка (strelka):** ещё одна классификация, например: niedкоэтажная, среднеэтажная, высокоэтажная современная микрорайон и др.

- **Что покажет:** к какому типу городской застройки относится каждый квартал. Поможет понять общую структуру города и выявить однородные или разнородные зоны.

### 7. Доступность территории (Area accessibility)

- **area_accessibility** — средняя доступность квартала (коэффициент, без размерности)

- **Что покажет:** насколько легко из данного квартала добраться до других частей города с учётом времени в пути и площади кварталов.


## 🏗️ Индикаторы плотности застройки — подробный разбор

Считаются для каждого квартала отдельно. Все коэффициенты без размерности (от 0 до 1 или больше), что позволяет сравнивать кварталы разного размера.

| Показатель | Что считает (простыми словами) | Единица | Что покажет о Светлогорске |
|------------|-------------------------------|---------|---------------------------|
| **site_area** | Площадь всего квартала | кв.м | Как besar квартал — от этого зависят все остальные коэффициенты |
| **FSI** | Суммарная площадь всех этажей всех зданий ÷ площадь квартала | коэф. | Сколько «слоёв» зданий на территории. FSI = 1 — общая площадь этажей равна площади квартала. FSI = 2 — в два раза больше (в среднем двухэтажная застройка) |
| **GSI** | Площадь застройки зданий (отпечаток) ÷ площадь квартала | коэф. | Какая доля квартала занята зданиями в плане. GSI = 0.2 — 20% территории застроено, остальное — дворы, улицы, зелёные зоны |
| **MXI** | Жилая площадь ÷ общая площадь этажей | коэф. | Какая часть площади зданий жилая. MXI = 0.8 — 80% площади зданий жилая, 20% — нежилая (магазины, офисы) |
| **L** | Общая площадь этажей ÷ площадь застройки | коэф. | Средняя этажность застройки. L = 5 — дома в квартале в среднем пятиэтажные |
| **OSR** | Открытое пространство ÷ площадь квартала | коэф. | Сколько территории остаётся свободным (дворы, парки, парковки). OSR = 0.7 — 70% квартала не застроено |
| **share_living** | Жилая площадь ÷ площадь квартала | коэф. | Какая доля территории квартала приходится на жилую площадь зданий |
| **share_non_living** | Нежилая площадь ÷ площадь квартала | коэф. | Какая доля территории квартала приходится на нежилую площадь зданий |

### Пример:

Допустим, в квартале:
- Площадь квартала: 10 000 кв.м
- Площадь застройки зданий: 2 000 кв.м → **GSI = 0.2** (20% территории под домами)
- Суммарная площадь всех этажей: 10 000 кв.м → **FSI = 1.0**
- Средняя этажность: 10 000 / 2 000 = **L = 5** (дома в среднем пятиэтажные)
- Жилая площадь: 8 000 кв.м → **MXI = 0.8** (80% площади этажей — жильё)
- Свободная территория: 8 000 кв.м → **OSR = 0.8** (80% квартала — дворы, улицы, зелень)


## 2. Назначение земельных 이용ов (Land use assignment)

Определение функционального назначения каждой зоны на основе данных OpenStreetMap.
Кварталы получают столбец `land_use` (доминирующий тип использования) и `share` (доля территории).

In [ ]:
# =============================================================================
# 2. Назначение земельных 이용ов
# =============================================================================

if BLOCKS_LANDUSE_PATH.exists():
    print(f"Загрузка существующих кварталов с назначением из {BLOCKS_LANDUSE_PATH}")
    blocks = gpd.read_file(BLOCKS_LANDUSE_PATH)
else:
    functional_zones = ox.features_from_polygon(
        boundaries.to_crs(4326).union_all(), tags={"landuse": True}
    )
    functional_zones = (
        functional_zones.reset_index(drop=True)[["geometry", "landuse"]]
        .rename(columns={"landuse": "functional_zone"})
        .to_crs(crs)
    )

    rules = {
        "commercial": LandUse.BUSINESS,
        "industrial": LandUse.INDUSTRIAL,
        "cemetery": LandUse.SPECIAL,
        "garages": LandUse.INDUSTRIAL,
        "residential": LandUse.RESIDENTIAL,
        "retail": LandUse.BUSINESS,
        "grass": LandUse.RECREATION,
        "farmland": LandUse.AGRICULTURE,
        "construction": LandUse.SPECIAL,
        "brownfield": LandUse.INDUSTRIAL,
        "forest": LandUse.RECREATION,
        "recreation_ground": LandUse.RECREATION,
        "religious": LandUse.SPECIAL,
        "flowerbed": LandUse.RECREATION,
        "military": LandUse.SPECIAL,
        "landfill": LandUse.TRANSPORT,
    }

    blocks = assign_land_use(
        blocks,
        functional_zones.reset_index(drop=True).rename(columns={"landuse": "functional_zone"}),
        rules,
    )
    blocks.to_file(BLOCKS_LANDUSE_PATH, driver="GeoJSON")
    print(f"Кварталы с назначением сохранены в {BLOCKS_LANDUSE_PATH}")

blocks.head()

In [ ]:
# Визуализация кварталов с назначением земельных 이용ов
ax = blocks.plot(color='#ddd')
blocks.plot(column='land_use', legend=True, ax=ax).set_axis_off()

## 3. Расчёт матрицы доступности (Accessibility matrix)

### Источник графа

Граф загружается автоматически из OpenStreetMap через библиотеку `iduedu` (входит в состав BlocksNet).
Тип графа: **пешеходный (`walk`)**. Интермодальный граф не строится.

Единица времени в матрице: **минуты**.

In [ ]:
# =============================================================================
# 3.1. Построение пешеходного графа и матрицы доступности
# =============================================================================

print(f"Построение пешеходного графа для территории: {TERRITORY_NAME}")
print(f"Тип графа: {GRAPH_TYPE}")
print(f"Источник графа: OpenStreetMap (через iduedu / blocksnet.relations)")

graph = get_accessibility_graph(boundaries, graph_type=GRAPH_TYPE)

print(f"Узлов в графе: {graph.number_of_nodes()}")
print(f"Рёбер в графе: {graph.number_of_edges()}")

print("Расчёт матрицы времени в пути между кварталами...")
acc_mx = calculate_accessibility_matrix(blocks, graph)

# Проверка единиц измерения — время в минутах
print(f"Размер матрицы: {acc_mx.shape}")
print(f"Единица измерения: {TIME_UNIT}")
acc_mx.head()

In [ ]:
# Сохранение матрицы доступности
acc_mx.to_csv(ACCESSIBILITY_MX_PATH, index=True)
print(f"Матрица доступности сохранена в {ACCESSIBILITY_MX_PATH}")

## 4. Агрегация зданий в кварталы (Buildings aggregation)

Загружаем здания из OpenStreetMap, заполняем отсутствующие данные и агрегируем параметры зданий по кварталам:
- `population` — население
- `footprint_area` — площадь застройки
- `build_floor_area` — общая площадь этажей
- `living_area`, `non_living_area` — жилая и нежилая площадь
- `objects_count` — количество объектов в квартале

In [ ]:
# =============================================================================
# 4. Агрегация зданий в кварталы
# =============================================================================

if BUILDINGS_BLOCKS_PATH.exists():
    print(f"Загрузка существующих агрегированных зданий из {BUILDINGS_BLOCKS_PATH}")
    buildings_blocks = gpd.read_file(BUILDINGS_BLOCKS_PATH)
else:
    buildings_path = DATA_DIR / "buildings.geojson"

    if buildings_path.exists():
        buildings = gpd.read_file(buildings_path)
        print(f"Здания загружены из {buildings_path}")
    else:
        print("Загрузка зданий из OpenStreetMap...")
        buildings = ox.features_from_polygon(
            boundaries.to_crs(4326).union_all(), tags={"building": True}
        ).reset_index(drop=True).to_crs(crs)
        buildings.to_file(buildings_path, driver="GeoJSON")
        print(f"Здания сохранены в {buildings_path}")

    # is_living: жилые здания
    is_living_tags = ["residential", "house", "apartments", "detached", "terrace", "dormitory"]
    buildings["is_living"] = buildings["building"].apply(lambda b: b in is_living_tags)

    # number_of_floors
    buildings["number_of_floors"] = pd.to_numeric(buildings["building:levels"], errors="coerce")

    # Imputation missing data
    buildings = impute_buildings(buildings, default_living_demand=30)
    buildings.to_file(BUILDINGS_PATH, driver="GeoJSON")

    # Aggregation within blocks
    buildings_blocks = aggregate_objects(blocks, buildings)[0]
    buildings_blocks.to_file(BUILDINGS_BLOCKS_PATH, driver="GeoJSON")
    print(f"Агрегированные здания сохранены в {BUILDINGS_BLOCKS_PATH}")

buildings_blocks.head()

In [ ]:
# Визуализация населения по кварталам
buildings_blocks.plot('population', legend=True).set_axis_off()

## 5. Агрегация сервисов в кварталы (Services aggregation)

Пример: загрузка школ (`amenity=school`), заполнение пропусков и агрегация по кварталам:
- `capacity` — вместимость
- `objects_count` — количество объектов в квартале

In [ ]:
# =============================================================================
# 5. Агрегация сервисов в кварталы
# =============================================================================

if SERVICES_BLOCKS_PATH.exists():
    print(f"Загрузка существующих агрегированных сервисов из {SERVICES_BLOCKS_PATH}")
    services_blocks = gpd.read_file(SERVICES_BLOCKS_PATH)
else:
    services_path = DATA_DIR / "services.geojson"

    if services_path.exists():
        services = gpd.read_file(services_path)
        print(f"Сервисы загружены из {services_path}")
    else:
        print("Загрузка сервисов из OpenStreetMap (школы)...")
        services = ox.features_from_polygon(
            boundaries.to_crs(4326).union_all(), tags={"amenity": "school"}
        ).reset_index(drop=True).to_crs(crs)
        services.to_file(services_path, driver="GeoJSON")
        print(f"Сервисы сохранены в {services_path}")

    # Imputation missing data
    services = impute_services(services, "school")

    # Aggregation within blocks
    services_blocks = aggregate_objects(blocks, services)[0]
    services_blocks.to_file(SERVICES_BLOCKS_PATH, driver="GeoJSON")
    print(f"Агрегированные сервисы сохранены в {SERVICES_BLOCKS_PATH}")

services_blocks.head()

In [ ]:
# Визуализация вместимости школ по кварталам
services_blocks.plot('capacity', legend=True).set_axis_off()

## 6. Расчёт всех показателей (Results)

Расчёт показан для примера с сервисом «школа». Если нужны другие сервисы или метрики — укажите их отдельно.

In [ ]:
# =============================================================================
# 6.1. Конкурентное обеспечение (competitive_provision)
# =============================================================================

if PROVISION_PATH.exists() and LINKS_PATH.exists():
    print(f"Загрузка существующего обеспечения из {PROVISION_PATH}")
    prov_df = pd.read_csv(PROVISION_PATH)
    links_df = pd.read_csv(LINKS_PATH)
else:
    df = buildings_blocks[["population"]].join(services_blocks[["capacity"]])

    _, demand, accessibility = service_types_config["school"].values()

    prov_df, links_df = competitive_provision(df, acc_mx, accessibility, demand)
    print(f"PROVISION_PATH = {PROVISION_PATH}")
    print(f"LINKS_PATH = {LINKS_PATH}")
    prov_df.to_csv(PROVISION_PATH, index=False)
    print(f"Обеспечение сохранено в {PROVISION_PATH}")
    links_df.to_csv(LINKS_PATH, index=True)
    print(f"Связи между кварталами сохранены в {LINKS_PATH}")

prov_df.head()

In [ ]:
# Визуализация обеспечения школами на карте
ax = blocks.plot(color='#ddd')
blocks.join(prov_df).plot('provision_strong', ax=ax, cmap='RdYlGn', vmin=0, vmax=1, legend=True)
ax.set_axis_off()

# Показать связи между кварталами (первые строки)
if 'links_df' in globals():
    print(links_df.head())

In [ ]:
# =============================================================================
# 6.2. Плотностные индикаторы (calculate_density_indicators)
# =============================================================================

if DENSITY_PATH.exists():
    print(f"Загрузка существующих индикаторов из {DENSITY_PATH}")
    density_df = pd.read_csv(DENSITY_PATH)
else:
    buildings_blocks["site_area"] = buildings_blocks.area
    density_df = calculate_density_indicators(buildings_blocks)
    density_df.to_csv(DENSITY_PATH, index=False)
    print(f"Индикаторы плотности сохранены в {DENSITY_PATH}")

density_df.head()

In [ ]:
# =============================================================================
# 6.3. Морфотипы (spacematrix + strelka)
# =============================================================================

if MORPHOTYPES_PATH.exists():
    print(f"Загрузка существующих морфотипов из {MORPHOTYPES_PATH}")
    spacematrix_df = pd.read_csv(MORPHOTYPES_PATH)
else:
    spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)
    strelka_df = get_strelka_morphotypes(density_df)
    morphotypes = spacematrix_df.copy()
    morphotypes["morphotype_strelka"] = strelka_df["morphotype"]
    morphotypes.to_csv(MORPHOTYPES_PATH, index=False)
    print(f"Морфотипы сохранены в {MORPHOTYPES_PATH}")

spacematrix_df.head()

In [ ]:
# Визуализация морфотипов пространственной матрицы
ax = blocks.plot(color='#ddd')
blocks.join(spacematrix_df).plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()

# Визуализация морфотипов Стрелка
ax = blocks.plot(color='#ddd')
blocks.join(spacematrix_df[['morphotype_strelka']]).plot('morphotype_strelka', legend=True, ax=ax)
ax.set_axis_off()

In [ ]:
# =============================================================================
# 6.4. Доступность территории (area_accessibility)
# =============================================================================

if AREA_ACCESSIBILITY_PATH.exists():
    print(f"Загрузка существующей доступности из {AREA_ACCESSIBILITY_PATH}")
    area_acc = pd.read_csv(AREA_ACCESSIBILITY_PATH)
else:
    # Убеждаемся, что site_area есть в buildings_blocks
    if "site_area" not in buildings_blocks.columns:
        buildings_blocks["site_area"] = buildings_blocks.area
    
    # Выравниваем индексы: оба на целые числа 0..N-1
    blocks_reset = buildings_blocks.reset_index(drop=True).copy()
    acc_mx_reset = acc_mx.reset_index(drop=True).copy()
    acc_mx_reset.columns = list(range(acc_mx_reset.shape[1]))
    blocks_reset.index = list(range(len(blocks_reset)))
    
    # Проверяем, что размеры совпадают
    if len(blocks_reset) != acc_mx_reset.shape[0]:
        raise ValueError(
            f"Количество кварталов ({len(blocks_reset)}) не совпадает с размером матрицы ({acc_mx_reset.shape[0]})"
        )
    
    area_acc = area_accessibility(acc_mx_reset, blocks_reset)
    area_acc.to_csv(AREA_ACCESSIBILITY_PATH, index=False)
    print(f"Доступность территории сохранена в {AREA_ACCESSIBILITY_PATH}")

area_acc.head()

In [ ]:
# Визуализация доступности территории
blocks.join(area_acc).plot('area_accessibility', legend=True).set_axis_off()

## 7. Проверка результатов и повторный запуск

### Инструкция повторного запуска

1. Активируйте виртуальное окружение (PowerShell):
   ```powershell
   .venv\Scripts\Activate.ps1
   ```

2. Убедитесь, что `blocksnet` установлен (версия 1.0.0.a9):
   ```powershell
   pip show blocksnet
   ```

3. Запустите Jupyter Notebook:
   ```powershell
   jupyter notebook examples/svetlogorsk_pipeline.ipynb
   ```

4. Запускайте ячейки по порядку. Промежуточные результаты сохраняются в `data/` и `results/`;
   при повторном запуске существующие файлы используются повторно (не перекачиваются).

### Проверка результатов

- `results/boundary.geojson` — граница территории Светлогорска
- `results/blocks.geojson` — городские кварталы (остановка ⏸)
- `results/blocks_landuse.geojson` — кварталы с назначением земельных 이용ов
- `results/accessibility_matrix.csv` — матрица времени в пути (минуты, пешеходный граф)
- `results/buildings_blocks.geojson` — агрегированные здания по кварталам
- `results/services_blocks.geojson` — агрегированные сервисы по кварталам
- `results/provision.csv` — конкурентное обеспечение
- `results/density_indicators.csv` — плотностные индикаторы
- `results/morphotypes.csv` — морфотипы
- `results/area_accessibility.csv` — доступность территории

Если все файлы созданы и содержат данные, исследование выполнено корректно.